# AI Audio Detector — Data Preparation

Collects and organises **all** training data for the AI audio detector.
Run this notebook **before** `train_colab.ipynb`.

**What this notebook does:**

| Section | Data | Label | Source |
|---------|------|-------|--------|
| A | Real non-human sounds | 0 (real) | ESC-50 (your ZIP) |
| B | AI non-human sounds | 1 (AI) | AudioGen generated in Colab |
| C | Real voice | 0 (real) | LJSpeech download (~2.6 GB) |
| D | AI voice | 1 (AI) | WaveFake melgan download (~7 GB) |
| E | Build manifest | — | `run_preprocessing()` |

**Before running:**
1. Runtime → Change runtime type → **GPU** (T4 is fine)
2. Upload `ESC-50-master.zip` to Google Drive at:
   `My Drive/AI-Innovation-Data/raw/non_human/real/ESC-50-master.zip`
3. Run cells top to bottom

Total time: ~3–4 hours (mostly Section B AudioGen generation + WaveFake download)


In [1]:
import os, shutil, sys, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    REPO_DIR   = '/content/AI-Innovation'
    DRIVE_ROOT = '/content/drive/MyDrive/AI-Innovation-Data'
    BRANCH     = 'AI_sounds'

    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
        CLONE_URL = f'https://{token}@github.com/foojanbabaeeian/AI-Innovation.git'
        print('Using GitHub token from Colab Secrets.')
    except Exception:
        CLONE_URL = 'https://github.com/foojanbabaeeian/AI-Innovation.git'

    if not os.path.exists(REPO_DIR):
        print('Cloning repository...')
        ret = os.system(f'git clone --branch {BRANCH} "{CLONE_URL}" {REPO_DIR}')
        if ret != 0 or not os.path.exists(REPO_DIR):
            raise RuntimeError('Clone failed -- make repo public or add GITHUB_TOKEN to Colab Secrets.')
    else:
        os.system(f'cd {REPO_DIR} && git pull')

    os.chdir(REPO_DIR)
    sys.path.insert(0, REPO_DIR)

    for link, target in [('data/raw', f'{DRIVE_ROOT}/raw'), ('outputs', f'{DRIVE_ROOT}/checkpoints')]:
        Path(target).mkdir(parents=True, exist_ok=True)
        link_path = Path(os.getcwd()) / link
        link_path.parent.mkdir(parents=True, exist_ok=True)
        if link_path.is_symlink(): link_path.unlink()
        elif link_path.exists(): shutil.rmtree(link_path)
        os.symlink(target, link_path)
        print(f'  {link} -> {target}')

else:
    # LOCAL: Google Drive Streaming path
    DRIVE_ROOT = 'C:/Users/fooja/Google Drive Streaming/My Drive/AI-Innovation-Data'

    _p = Path.cwd()
    while _p != _p.parent:
        if (_p / 'src').exists() and (_p / 'requirements.txt').exists():
            break
        _p = _p.parent
    REPO_DIR = str(_p)
    os.chdir(REPO_DIR)
    sys.path.insert(0, REPO_DIR)

    # Windows directory junction (no admin rights needed)
    for link, target in [('data/raw', f'{DRIVE_ROOT}/raw'), ('outputs', f'{DRIVE_ROOT}/checkpoints')]:
        Path(target).mkdir(parents=True, exist_ok=True)
        link_path = Path(REPO_DIR) / link
        link_path.parent.mkdir(parents=True, exist_ok=True)
        # Remove existing junction/symlink/dir safely (is_junction requires Python 3.12+)
        if link_path.exists() or link_path.is_symlink():
            try:
                os.rmdir(str(link_path))   # removes junction or empty dir without touching target
            except OSError:
                shutil.rmtree(link_path)   # non-empty real directory
        subprocess.run(['cmd', '/c', 'mklink', '/J', str(link_path), target.replace('/', os.sep)], check=True)
        print(f'  {link} -> {target}')

os.makedirs('data/metadata', exist_ok=True)
env_name = 'Colab' if IN_COLAB else 'Local'
print(f'Environment : {env_name}')
print(f'Working dir : {os.getcwd()}')
print(f'Drive root  : {DRIVE_ROOT}')
print('Setup complete.')


  data/raw -> C:/Users/fooja/Google Drive Streaming/My Drive/AI-Innovation-Data/raw
  outputs -> C:/Users/fooja/Google Drive Streaming/My Drive/AI-Innovation-Data/checkpoints
Environment : Local
Working dir : c:\Users\fooja\Documents\GitHub\AI-Innovation
Drive root  : C:/Users/fooja/Google Drive Streaming/My Drive/AI-Innovation-Data
Setup complete.


In [2]:
import os, subprocess, sys

if IN_COLAB:
    os.system('pip install -q -r requirements.txt')
    os.system('apt-get install -q -y ffmpeg libsndfile1')
else:
    # Locally: pip install only what might be missing (audiocraft is the key one)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
    print('Tip: make sure ffmpeg is on your PATH (choco install ffmpeg)')

import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'
print(f'PyTorch {torch.__version__}  |  GPU: {gpu}')


KeyboardInterrupt: 

---
## Section A — Real non-human sounds: ESC-50

Upload `ESC-50-master.zip` to Drive at:
`My Drive/AI-Innovation-Data/raw/non_human/real/ESC-50-master.zip`

Then run the cell below to extract and ingest it.


In [ ]:
# -----------------------------------------------------------------
# Cell 3 -- Download and ingest ESC-50 (via git-lfs clone)
#
# The GitHub "Download ZIP" gives LFS pointer stubs, not real audio.
# We clone directly with git-lfs to get the actual WAV files.
# -----------------------------------------------------------------
import os, sys
from pathlib import Path

ESC50_CLONE = '/content/ESC-50'

# Install git-lfs if needed
os.system('apt-get install -q -y git-lfs && git lfs install')

if not Path(ESC50_CLONE).exists():
    print('Cloning ESC-50 with git-lfs (real audio files, ~600 MB)...')
    ret = os.system(f'git lfs clone https://github.com/karolpiczak/ESC-50.git {ESC50_CLONE}')
    if ret != 0:
        raise RuntimeError('ESC-50 clone failed. Check your internet connection.')
else:
    print('ESC-50 already cloned.')

# Verify we got real audio (LFS files >10 KB; pointer stubs are ~130 bytes)
sample_wavs = list(Path(f'{ESC50_CLONE}/audio').glob('*.wav'))[:3]
for w in sample_wavs:
    size = w.stat().st_size
    if size < 1000:
        raise RuntimeError(
            f'{w.name} is only {size} bytes -- still an LFS pointer.'
            'git-lfs clone may have failed silently. Re-run this cell.'
        )
print(f'Audio files look real: {sample_wavs[0].name} = {sample_wavs[0].stat().st_size / 1024:.0f} KB')

# Ingest into the data pipeline
from src.data.ingestors.audioset import ingest_esc50
n = ingest_esc50(
    source_dir=ESC50_CLONE,
    output_dir='data/raw/non_human/real/esc50_processed',
    copy=False,
    non_human_only=True,
)
print(f'ESC-50 ingested: {n} clips (non-human categories only)')

Cloning ESC-50 with git-lfs (real audio files, ~600 MB)...
Audio files look real: 1-100032-A-0.wav = 431 KB
ESC-50 ingested: 2000 clips (non-human categories only)


---
## Section B — AI non-human sounds: AudioGen

Uses Meta's AudioGen model to generate 200 AI environmental sound clips.
Each clip is 5 seconds at 16kHz. ~2–3 hours on T4 GPU.

These clips are **definitively AI-generated** (label=1), with no copyright issues.


In [ ]:
import sys, subprocess, os

# av must be installed via conda on Windows (no pre-built pip wheel)
# Run this once in Anaconda Prompt if not done yet:
#   conda activate tf-gpu-210
#   conda install -c conda-forge av ffmpeg -y

try:
    import av
    print(f'av {av.__version__} already installed.')
except ImportError:
    print('av not found -- trying conda install (may take a moment)...')
    ret = subprocess.run(
        ['conda', 'install', '-c', 'conda-forge', 'av', 'ffmpeg', '-y', '--quiet'],
        capture_output=True, text=True
    )
    if ret.returncode != 0:
        raise RuntimeError(
            'conda install av failed.
'
            'Run manually in Anaconda Prompt:
'
            '  conda activate tf-gpu-210
'
            '  conda install -c conda-forge av ffmpeg -y'
        )
    print('av installed via conda.')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'audiocraft'], check=True)
print('AudioCraft installed.')


In [4]:
# ─────────────────────────────────────────────────────────────────
# Cell 5 — Generate 200 AI environmental sounds with AudioGen
# ─────────────────────────────────────────────────────────────────
import torch, torchaudio, os
from pathlib import Path
from audiocraft.models import AudioGen
from audiocraft.data.audio import audio_write

AI_SFX_OUT = Path('data/raw/non_human/fake/audiogen')
AI_SFX_OUT.mkdir(parents=True, exist_ok=True)

# 200 diverse environmental sound prompts (AI-generated counterparts to ESC-50)
PROMPTS = [
    # Rain & water
    "heavy rain on a tin roof", "light drizzle on pavement", "tropical downpour",
    "rain falling on leaves", "thunderstorm with heavy rain", "hailstorm on glass",
    "ocean waves crashing on rocks", "gentle ocean surf", "river flowing over rocks",
    "small stream babbling", "waterfall in a forest", "water dripping in a cave",
    "lake water lapping shore", "waves on a pebble beach", "rain in a forest",
    # Thunder & wind
    "loud thunder clap", "distant rolling thunder", "thunder rumble after lightning",
    "strong wind gusts", "wind howling through trees", "gentle breeze in leaves",
    "wind in tall grass", "gusty wind through a canyon", "stormy wind at night",
    "wind whistling through cracks",
    # Fire
    "crackling campfire", "roaring bonfire", "fire in a fireplace",
    "small fire burning dry wood", "burning leaves", "fire popping and hissing",
    # Animals
    "dog barking in distance", "dog growling", "multiple dogs barking",
    "cat meowing", "cat purring", "rooster crowing at dawn",
    "cow mooing in a field", "horse neighing", "sheep bleating",
    "pig oinking", "frog croaking at night", "crickets at night",
    "cicadas in summer heat", "birds chirping in morning", "owl hooting at night",
    "crow cawing", "hen clucking", "duck quacking",
    "wolf howling at moon", "bear growling",
    # Insects
    "buzzing bee", "mosquito buzzing", "grasshoppers in field",
    "insect sounds at night", "cicadas and crickets together",
    # Mechanical / urban
    "car engine idling", "car horn honking", "car passing on highway",
    "motorcycle engine revving", "train passing on tracks",
    "helicopter flying overhead", "airplane flying above", "jet engine roar",
    "ambulance siren", "police car siren", "fire truck siren",
    "chainsaw running", "electric drill", "jackhammer on concrete",
    "lawnmower", "vacuum cleaner running", "washing machine cycle",
    "washing machine spin", "refrigerator humming", "air conditioner",
    # Indoor / everyday
    "keyboard typing fast", "mouse clicking", "clock ticking",
    "clock alarm ringing", "phone ringing", "door knocking",
    "door creaking open", "glass shattering", "dishes clattering",
    "keys jingling", "coins dropping on floor", "paper rustling",
    "book pages turning", "scissors cutting paper",
    # Crowd & ambience
    "crowd cheering at stadium", "crowd applause", "busy street ambience",
    "restaurant background noise", "market crowd sounds",
    "construction site ambience", "airport terminal ambience",
    "subway train arriving", "school playground sounds",
    # Explosions / impacts
    "fireworks exploding", "fireworks show", "single firework pop",
    "balloon popping", "book dropped on floor", "door slamming shut",
    # Nature ambience
    "tropical jungle ambience", "forest ambience at dawn",
    "forest ambience at night", "desert wind ambience",
    "mountain stream with birds", "swamp ambience with frogs",
    "summer meadow with insects", "winter wind in bare trees",
    "spring rain on grass", "autumn leaves blowing",
    # Unique weather
    "blizzard snow and wind", "fog horn in harbor",
    "hail on car roof", "ice cracking",
    # Additional animal sounds
    "lion roaring", "elephant trumpeting", "monkey chattering",
    "dolphin clicking", "whale song", "seagulls calling",
    "geese honking", "woodpecker drumming on tree",
    "turkey gobbling", "peacock calling",
    # Water continued
    "toilet flushing", "sink water running", "shower running",
    "kettle boiling", "popcorn popping", "bacon sizzling",
    "ice cubes in glass", "fizzy drink opening",
    # Vehicle continued
    "bus engine starting", "bus doors opening", "train whistle",
    "boat engine on water", "bicycle bell ringing",
    "skateboard rolling on pavement", "rollerskates on floor",
    # More nature
    "thunder and heavy rain together", "hail and thunder",
    "wind and rain at sea", "storm on open ocean",
    "stream with frogs at dusk", "birds and stream together",
    "forest fire crackling", "dry leaves in wind",
    # Electronics / signals
    "microwave beeping", "oven timer", "smoke alarm beeping",
    "camera shutter", "notification sound",
    # Sports / recreation
    "basketball bouncing", "tennis ball hit", "golf club swing",
    "baseball bat hit", "football crowd roar",
    # Tools
    "hammer hitting nail", "saw cutting wood", "welding torch",
    "electric sander", "power tool drilling",
    # Final ambiences
    "morning birds with light wind", "evening crickets with frogs",
    "midnight rain on rooftop", "sunrise bird chorus",
    "thunderstorm approaching", "storm passing with rain",
    "after rain bird sounds", "snow falling silently",
]

# Trim to exactly 200
PROMPTS = PROMPTS[:200]
assert len(PROMPTS) == 200, f'Expected 200 prompts, got {len(PROMPTS)}'

print(f'Loading AudioGen model...')
model = AudioGen.get_pretrained('facebook/audiogen-medium')
model.set_generation_params(duration=5)  # 5-second clips
print('Model loaded.')

# Check how many we already have
existing = list(AI_SFX_OUT.glob('*.wav'))
start_idx = len(existing)
print(f'Already generated: {start_idx}/200')

# Generate in batches of 4 (fits T4 GPU memory)
BATCH_SIZE = 4
generated = start_idx

for batch_start in range(start_idx, len(PROMPTS), BATCH_SIZE):
    batch_prompts = PROMPTS[batch_start:batch_start + BATCH_SIZE]

    with torch.no_grad():
        wavs = model.generate(batch_prompts)  # (B, 1, samples)

    for i, wav in enumerate(wavs):
        idx = batch_start + i
        out_path = AI_SFX_OUT / f'audiogen_{idx:04d}.wav'
        # Save as 16kHz mono WAV
        wav_16k = torchaudio.functional.resample(wav.cpu(), model.sample_rate, 16000)
        torchaudio.save(str(out_path), wav_16k, 16000)

    generated += len(batch_prompts)
    if batch_start % 40 == 0:
        print(f'  Generated {generated}/200 clips...')
        torch.cuda.empty_cache()

final_count = len(list(AI_SFX_OUT.glob('*.wav')))
print(f'\nDone! {final_count} AI SFX clips saved to {AI_SFX_OUT}')

ModuleNotFoundError: No module named 'audiocraft'

---
## Section C — Real voice: LJSpeech

LJSpeech is a single-speaker English audiobook dataset by Keith Ito.
13,100 utterances (about 24 hours), 22kHz WAV, ~2.6 GB.
Label = 0 (real human voice).


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cell 6 — Download and ingest LJSpeech (real voice)
# ─────────────────────────────────────────────────────────────────
import urllib.request, tarfile

LJSPEECH_TAR  = f'{DRIVE_ROOT}/raw/voice/real/LJSpeech-1.1.tar.bz2'
LJSPEECH_DIR  = f'{DRIVE_ROOT}/raw/voice/real/LJSpeech-1.1'

if Path(LJSPEECH_DIR).exists():
    print('LJSpeech already extracted.')
else:
    if not Path(LJSPEECH_TAR).exists():
        print('Downloading LJSpeech (~2.6 GB)...')
        Path(LJSPEECH_TAR).parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(
            'https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2',
            LJSPEECH_TAR,
        )
        print('Download complete.')

    print('Extracting...')
    with tarfile.open(LJSPEECH_TAR, 'r:bz2') as tf:
        tf.extractall(f'{DRIVE_ROOT}/raw/voice/real/')
    print('Extracted.')

from src.data.ingestors.voice import ingest_ljspeech
n = ingest_ljspeech(
    source_dir=LJSPEECH_DIR,
    output_dir='data/raw/voice',
    copy=False,
)
print(f'LJSpeech ingested: {n} files (real voice, label=0)')

---
## Section D — AI voice: WaveFake (MelGAN vocoder)

WaveFake is a deepfake speech detection benchmark that regenerates LJSpeech
utterances using 6 different neural vocoders.  We download just the MelGAN
generator (~7 GB) to save time.

Label = 1 (AI-generated voice).


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cell 7 — Download WaveFake melgan from Zenodo
# ─────────────────────────────────────────────────────────────────
import requests, zipfile, io

WAVEFAKE_OUT = Path(f'{DRIVE_ROOT}/raw/voice/fake/wavefake')
WAVEFAKE_OUT.mkdir(parents=True, exist_ok=True)

# Dynamically get file list from Zenodo API
print('Querying Zenodo for WaveFake files...')
resp = requests.get('https://zenodo.org/api/records/5642694', timeout=30)
resp.raise_for_status()
zenodo = resp.json()

# Find the melgan zip
melgan_file = None
for f in zenodo.get('files', []):
    if 'melgan' in f['key'].lower() and f['key'].endswith('.zip'):
        if 'large' not in f['key'].lower() and 'multi' not in f['key'].lower():
            melgan_file = f
            break

if melgan_file is None:
    # Fallback: print all files for the user to choose
    print('Available WaveFake files:')
    for f in zenodo.get('files', []):
        size_gb = f.get('size', 0) / 1e9
        print(f"  {f['key']}  ({size_gb:.1f} GB)  → {f['links']['self']}")
    print()
    print('Set DOWNLOAD_URL below and re-run:')
    DOWNLOAD_URL = ''
else:
    size_gb = melgan_file.get('size', 0) / 1e9
    DOWNLOAD_URL = melgan_file['links']['self']
    print(f"Found: {melgan_file['key']}  ({size_gb:.1f} GB)")
    print(f'URL: {DOWNLOAD_URL}')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cell 8 — Download and extract WaveFake melgan
# ─────────────────────────────────────────────────────────────────
# NOTE: If Cell 7 printed a list of URLs, paste the melgan one here:
# DOWNLOAD_URL = 'https://zenodo.org/records/5642694/files/ljspeech_melgan.zip'

if not DOWNLOAD_URL:
    print('No URL set. See Cell 7 output and set DOWNLOAD_URL.')
else:
    melgan_zip = WAVEFAKE_OUT / 'ljspeech_melgan.zip'
    melgan_dir = WAVEFAKE_OUT / 'ljspeech_melgan'

    if melgan_dir.exists() and len(list(melgan_dir.glob('*.wav'))) > 0:
        print(f'WaveFake melgan already extracted: {len(list(melgan_dir.glob("*.wav")))} files')
    else:
        if not melgan_zip.exists():
            print('Downloading WaveFake melgan (this may take 10-20 min)...')
            !wget -q --show-progress -O {melgan_zip} "{DOWNLOAD_URL}"
            print('Download complete.')

        print('Extracting...')
        with zipfile.ZipFile(melgan_zip, 'r') as zf:
            zf.extractall(WAVEFAKE_OUT)
        print('Extracted.')

    # Ingest melgan as fake voice
    from src.data.ingestors.voice import ingest_generic_audio
    n = ingest_generic_audio(
        source_dir=str(melgan_dir),
        output_dir='data/raw/voice/fake/wavefake_melgan',
        label=1,
        source_name='wavefake_melgan',
        copy=False,
    )
    print(f'WaveFake melgan ingested: {n} files (AI voice, label=1)')

---
## Section D2 — AI voice backup: SpeechT5 (if WaveFake download too slow)

If the WaveFake download is taking too long, run this cell instead.
Generates 1,000 AI speech clips using Microsoft's SpeechT5 TTS model.
Uses LibriSpeech transcripts as input text.


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cell 9 — Generate AI voice with SpeechT5 (backup, ~45 min on T4)
# Skip this cell if WaveFake download succeeded.
# ─────────────────────────────────────────────────────────────────
SPEECHT5_OUT = Path('data/raw/voice/fake/speecht5')
existing_speecht5 = list(SPEECHT5_OUT.glob('*.wav'))

if len(existing_speecht5) >= 500:
    print(f'SpeechT5 already generated: {len(existing_speecht5)} files. Skipping.')
else:
    SPEECHT5_OUT.mkdir(parents=True, exist_ok=True)
    !pip install -q datasets

    from transformers import (
        SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
    )
    from datasets import load_dataset
    import torch, torchaudio, numpy as np

    print('Loading SpeechT5 model...')
    processor = SpeechT5Processor.from_pretrained('microsoft/speecht5_tts')
    tts_model = SpeechT5ForTextToSpeech.from_pretrained('microsoft/speecht5_tts').cuda()
    vocoder   = SpeechT5HifiGan.from_pretrained('microsoft/speecht5_hifigan').cuda()

    # Speaker embeddings (CMU Arctic dataset)
    print('Loading speaker embeddings...')
    embed_ds = load_dataset('Matthijs/cmu-arctic-xvectors', split='validation')
    # Use 5 different speakers for variety
    speaker_ids = [7306, 7307, 7308, 7309, 7310]
    speaker_embeds = [
        torch.tensor(embed_ds[sid]['xvector']).unsqueeze(0).cuda()
        for sid in speaker_ids
    ]

    # LibriSpeech-clean texts as input
    print('Loading LibriSpeech texts...')
    libri = load_dataset('librispeech_asr', 'clean', split='test', trust_remote_code=True)
    texts = [row['text'] for row in libri.select(range(min(1000, len(libri))))]

    print(f'Generating {len(texts)} TTS clips...')
    for i, text in enumerate(texts):
        out_path = SPEECHT5_OUT / f'speecht5_{i:05d}.wav'
        if out_path.exists():
            continue

        # Rotate through speakers
        spk = speaker_embeds[i % len(speaker_embeds)]
        try:
            inputs = processor(text=text[:200], return_tensors='pt').to('cuda')
            with torch.no_grad():
                speech = tts_model.generate_speech(inputs['input_ids'], spk, vocoder=vocoder)
            # Resample from 16kHz (SpeechT5 output) to 16kHz (already correct)
            torchaudio.save(str(out_path), speech.unsqueeze(0).cpu(), 16000)
        except Exception as e:
            pass  # Skip problematic texts

        if i % 100 == 0:
            print(f'  {i}/{len(texts)} clips generated...')
            torch.cuda.empty_cache()

    final_count = len(list(SPEECHT5_OUT.glob('*.wav')))
    print(f'SpeechT5 generation complete: {final_count} clips saved.')

---
## Section E — Build manifest and verify


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cell 10 — Verify data directories before building manifest
# ─────────────────────────────────────────────────────────────────
import os
from pathlib import Path

domains = {
    'voice/real':      'data/raw/voice/real',
    'voice/fake':      'data/raw/voice/fake',
    'non_human/real':  'data/raw/non_human/real',
    'non_human/fake':  'data/raw/non_human/fake',
}

AUDIO_EXTS = {'.wav', '.flac', '.mp3'}

print('Data directory summary:')
print('-' * 50)
total = 0
for name, path in domains.items():
    p = Path(path)
    if p.exists():
        n = sum(1 for f in p.rglob('*') if f.suffix.lower() in AUDIO_EXTS)
        total += n
        status = '✓' if n > 0 else '⚠  EMPTY'
        print(f'  {status}  {name:<20} {n:>6} clips')
    else:
        print(f'  ✗  {name:<20}  not found')

print('-' * 50)
print(f'Total: {total} audio files')

if total < 100:
    print('\n⚠  Very few files found. Check that data was downloaded/extracted correctly.')
else:
    print('\n✓  Enough data to build manifest.')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cell 11 — Build master_manifest.csv
# ─────────────────────────────────────────────────────────────────
from src.data.preprocessing import run_preprocessing

n_rows = run_preprocessing(
    raw_dir='data/raw',
    output_dir='data/processed',
    manifest_path='data/metadata/master_manifest.csv',
    train_ratio=0.70,
    val_ratio=0.15,
)
print(f'Manifest: {n_rows} rows written.')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cell 12 — Verify manifest balance
# ─────────────────────────────────────────────────────────────────
import pandas as pd

df = pd.read_csv('data/metadata/master_manifest.csv')

print('=== Manifest Summary ===')
print(f'Total rows: {len(df)}')
print()
print('By domain + label:')
print(df.groupby(['domain', 'label']).size().unstack(fill_value=0).to_string())
print()
print('By split:')
print(df.groupby(['split', 'label']).size().unstack(fill_value=0).to_string())
print()
print('By source dataset:')
print(df.groupby('source_dataset').size().sort_values(ascending=False).to_string())

# Sanity check: make sure train loader works
print()
print('Testing DataLoader...')
from src.data.dataset import build_dataloader
loader = build_dataloader(
    manifest_path='data/metadata/master_manifest.csv',
    data_root='data',
    split='train',
    batch_size=4,
    num_workers=0,
    max_samples=8,
)
if loader:
    batch = next(iter(loader))
    print(f'  waveform shape : {batch["waveform"].shape}')  # (4, 1, 64000)
    print(f'  ai_ratio       : {batch["ai_ratio"]}')
    print(f'  class_label    : {batch["class_label"]}')
    print('✓ DataLoader works.')
else:
    print('✗ DataLoader returned None — check manifest path and split.')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cell 13 — Copy manifest to Drive for persistence
# ─────────────────────────────────────────────────────────────────
import shutil

MANIFEST_DRIVE = f'{DRIVE_ROOT}/master_manifest.csv'
shutil.copy2('data/metadata/master_manifest.csv', MANIFEST_DRIVE)
print(f'Manifest saved to Drive: {MANIFEST_DRIVE}')
print()
print('✓ Data preparation complete!')
print('  Next step: open train_colab.ipynb and run all cells.')